# Lesson 6 — Eigenvalues, Eigenvectors, and Spectral Intuition
**SRAI — Statistics, Reasoning and Artificial Intelligence**  
**Book 1 · PU-B01-C06 · M1_N06 · candidate v0.1.0**  
Creator and author: Mbaye Kebe

This teaching candidate connects invariant directions to matrix powers, dynamic stability and covariance structure. It is not yet a published or owner-approved release. All datasets and sector coefficients below are **synthetic illustrations**, not empirical evidence.

Prerequisites: Lesson 5 vector spaces, bases, rank and projections; matrix multiplication; vector norms; basic Python. Work through the explanations as well as the cells. A successful run demonstrates the coded checks, not independent review of every claim.

## Learning objectives
1. Define an eigenpair with a **nonzero** eigenvector and interpret its geometry.
2. Use an orthonormal eigenbasis for a real symmetric matrix.
3. Compute matrix powers and recognize failure of diagonalization.
4. Estimate an eigenpair using residual-controlled power iteration and state its limitations.
5. Distinguish asymptotic stability, boundary behavior and transient growth.
6. Interpret covariance and sector modes without equating variance or model weights with policy value.

## Start here — VS Code and Google Colab
**VS Code:** extract the complete candidate ZIP, open its folder, create/select a Python 3.11+ environment, install `requirements.txt`, then open this notebook and choose that environment as its kernel. See `README.md` for exact Windows commands.

**Colab:** open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, select this `.ipynb`, connect to a Python runtime, and choose **Runtime → Run all**. The first code cell downloads the exact public `srai_math` wheel if no bundled wheel is present. No Drive mount, private account token or manual wheel upload is required. Internet access is required for that download and missing third-party dependencies.

The runtime is the **1.1.1rc1 prerelease**, not a newly claimed stable package. Its SHA-256 is verified before installation into a private temporary import location. A different already-loaded `srai_math` triggers a stop: restart the kernel, then Run all. The numerical-dependency cell installs only missing/out-of-range declared packages; if one is already imported, it asks for a restart before proceeding. Review setup cells before running code from any source.

Known runtime spectral limitations are explained in section 4 and guarded here. The source ZIP and exact wheel are included in the full package. Live VS Code/Colab rendering and the public download remain separate acceptance checks; consult the packaged validation report for actual results.

In [ ]:
# SRAI runtime: local wheel when available, otherwise a pinned public download.
from pathlib import Path
import hashlib, importlib, io, subprocess, sys, tempfile, urllib.request, zipfile

WHEEL_NAME = 'srai_math-1.1.1rc1-py3-none-any.whl'
WHEEL_SHA256 = '6e7033465ad3d9bf4650227a11be0380512a44fd476a83d5828ad4ec4f07e923'
WHEEL_URL = ('https://github.com/mbayekebe/srai-book-01-mathematical-foundations/'
             'releases/download/srai-math-v1.1.1rc1/' + WHEEL_NAME)

def ensure_srai_runtime():
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11 or later is required. Select a compatible kernel.")
    # Support VS Code kernels starting at package root or notebook directory.
    locations = [Path.cwd(), *list(Path.cwd().parents)[:2]]
    local = next((p for root in locations
                  for p in (root/'wheels'/WHEEL_NAME, root/WHEEL_NAME)
                  if p.is_file()), None)
    if local:
        data = local.read_bytes()
    else:
        try:
            request = urllib.request.Request(WHEEL_URL, headers={'User-Agent': 'SRAI-Notebook/1.0'})
            with urllib.request.urlopen(request, timeout=45) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError('Insecure download redirect rejected.')
                data = response.read(1024 * 1024 + 1)
        except Exception as exc:
            raise RuntimeError('SRAI runtime download unavailable. Check connectivity and that the '
                               'pinned release asset is published. No alternative version was installed. '
                               + WHEEL_URL) from exc
    if len(data) > 1024 * 1024 or hashlib.sha256(data).hexdigest() != WHEEL_SHA256:
        raise RuntimeError('SRAI wheel checksum mismatch. Stop; do not install this file.')
    previous = globals().get('_SRAI_PUBLIC_TARGET')
    loaded = [m for n,m in list(sys.modules.items()) if n=='srai_math' or n.startswith('srai_math.')]
    for module in loaded:
        origin = getattr(module, '__file__', None)
        if not previous or not origin or not Path(origin).resolve().is_relative_to(previous):
            raise RuntimeError('Another srai_math is loaded. Restart the kernel, then Run all.')
    if previous:
        target = previous
    else:
        work = Path(tempfile.mkdtemp(prefix='srai_runtime_'))
        wheel = work/WHEEL_NAME
        wheel.write_bytes(data)
        target = work/'site'
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index',
                               '--no-deps', '--no-compile', '--target', str(target), str(wheel)])
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for entry in archive.infolist():
            if not entry.is_dir() and entry.filename.startswith('srai_math/'):
                path = target/entry.filename
                if not path.is_file() or path.read_bytes()!=archive.read(entry):
                    raise RuntimeError('Installed runtime differs from verified wheel. Restart kernel.')
    if str(target) not in sys.path:
        sys.path.insert(0, str(target))
    importlib.invalidate_caches()
    import srai_math
    if not Path(srai_math.__file__).resolve().is_relative_to(target):
        raise RuntimeError('Unexpected package import path. Restart kernel.')
    print('SRAI runtime READY: verified 1.1.1rc1 (' + ('local wheel' if local else 'public download') + ')')
    return target

_SRAI_PUBLIC_TARGET = ensure_srai_runtime()

In [ ]:
# Install declared wheel dependencies only if absent or outside the supported range.
from importlib.metadata import version, PackageNotFoundError
from packaging.specifiers import SpecifierSet
import importlib.util
DEPENDENCIES = {
    'numpy': ('numpy', '>=1.26,<3'), 'scipy': ('scipy', '>=1.11,<2'),
    'sympy': ('sympy', '>=1.12,<2'), 'pandas': ('pandas', '>=2.1,<3'),
    'matplotlib': ('matplotlib', '>=3.8,<4'),
    'scikit-learn': ('sklearn', '>=1.3,<2'), 'networkx': ('networkx', '>=3.2,<4'),
}
needed=[]
for distribution, (module, bounds) in DEPENDENCIES.items():
    try:
        available = version(distribution) in SpecifierSet(bounds)
    except PackageNotFoundError:
        available = False
    if not available:
        if module in sys.modules:
            raise RuntimeError('Restart kernel before updating ' + distribution)
        needed.append(distribution + bounds)
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *needed])
    importlib.invalidate_caches()
for distribution, (_, bounds) in DEPENDENCIES.items():
    if version(distribution) not in SpecifierSet(bounds):
        raise RuntimeError('Dependency check failed: ' + distribution)
print('Numerical dependencies READY')

In [ ]:
import srai_math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from srai_math.utils import environment_info, set_seed
from srai_math.algebra import (
    eigendecomposition, symmetric_eigendecomposition, diagonalize,
    matrix_power_via_eigendecomposition, spectral_radius,
)
set_seed(42)
np.set_printoptions(precision=8, suppress=True)
display(environment_info())
print({name: version(name) for name in DEPENDENCIES})

## 1. Eigenpairs: directions transformed into multiples of themselves
For a square matrix $A$, an eigenpair consists of a scalar $\lambda$ and a vector $v\ne0$ satisfying
$$
Av=\lambda v.
$$
For real eigenpairs, a positive eigenvalue preserves orientation, a negative one reverses it, and a zero eigenvalue sends the vector to zero. Thus “keeps its direction” needs qualification. Multiplying an eigenvector by any nonzero scalar gives another eigenvector for the same eigenvalue.

For $A=\operatorname{diag}(3,1)$, the coordinate axes are eigendirections. The vector $(1,1)^T$ is **not** an eigenvector: its image $(3,1)^T$ is not a scalar multiple of it.

### Numerical acceptance
Floating-point equality needs a tolerance. We reject zero vectors, normalize the candidate vector, and use the scaled residual
$$
r=\frac{\|Au-\lambda u\|_2}{\|A\|_2+|\lambda|},\qquad u=\frac{v}{\|v\|_2}.
$$
When the denominator is zero, the residual numerator is used directly. A small residual verifies an approximate eigenpair, **not** that its eigenvalue is dominant. Near repeated eigenvalues, compare eigenspaces rather than signed columns.

In [ ]:
def eigenpair_residual(a, eigenvalue, eigenvector):
    """Scale-aware residual; reject nonfinite, nonsquare and zero-vector inputs."""
    a=np.asarray(a, dtype=complex)
    v=np.asarray(eigenvector, dtype=complex)
    if a.ndim != 2 or a.shape[0] != a.shape[1] or a.shape[0] == 0:
        raise ValueError('A must be nonempty and square')
    if v.ndim != 1 or len(v) != len(a):
        raise ValueError('Vector dimensions do not agree')
    if not np.isfinite(a).all() or not np.isfinite(v).all() or not np.isfinite(eigenvalue):
        raise ValueError('Inputs must be finite')
    scale=np.max(np.abs(v))
    if scale == 0:
        raise ValueError('Zero vector is not an eigenvector')
    u=v/scale
    u=u/np.linalg.norm(u)
    numerator=np.linalg.norm(a@u-eigenvalue*u)
    denominator=np.linalg.norm(a, 2)+abs(eigenvalue)
    return float(numerator/denominator if denominator else numerator)

def residual_power_iteration(a, initial=None, tol=1e-10, max_iter=10000):
    """Real-matrix teaching implementation; reports residual and iteration count.

    A small residual certifies an approximate eigenpair, not dominance.
    Suitable spectral separation and an appropriate start are required.
    """
    if np.iscomplexobj(a) or (initial is not None and np.iscomplexobj(initial)):
        raise ValueError('This reference iteration accepts real inputs only')
    a=np.asarray(a, dtype=float)
    if a.ndim != 2 or a.shape[0] != a.shape[1] or len(a)==0 or not np.isfinite(a).all():
        raise ValueError('A must be finite, nonempty and square')
    if not np.isfinite(tol) or tol <= 0 or not isinstance(max_iter, int) or max_iter < 1:
        raise ValueError('Positive tolerance and integer iteration limit required')
    v=np.ones(len(a)) if initial is None else np.asarray(initial, dtype=float).copy()
    if v.shape != (len(a),) or not np.isfinite(v).all() or np.max(np.abs(v))==0:
        raise ValueError('Initial vector must be finite, nonzero and dimension-compatible')
    v=v/np.max(np.abs(v)); v=v/np.linalg.norm(v)
    a_scale=np.max(np.abs(a))
    if a_scale==0:
        return {'value':0., 'vector':v, 'iterations':0, 'residual':0.}
    scaled=a/a_scale
    for iteration in range(1,max_iter+1):
        w=scaled@v
        norm=np.linalg.norm(w)
        if norm==0 or not np.isfinite(norm):
            raise ValueError('Iteration reached zero/nonfinite vector; choose another start')
        v=w/norm
        value=float(v@a@v)
        residual=eigenpair_residual(a,value,v)
        if residual<=tol:
            return {'value':value, 'vector':v, 'iterations':iteration, 'residual':residual}
    raise RuntimeError('No residual convergence within max_iter; no eigenpair accepted')

In [ ]:
A=np.diag([3.,1.])
values,vectors=eigendecomposition(A)
assert np.allclose(np.sort(values),[1.,3.])
assert all(eigenpair_residual(A,x,vectors[:,i])<1e-12 for i,x in enumerate(values))
assert eigenpair_residual(A,3.,[1.,1.])>0.1
try:
    eigenpair_residual(A,3.,[0.,0.])
except ValueError:
    print('Zero vector correctly rejected')
else:
    raise AssertionError('Zero-vector guard failed')
display(pd.DataFrame({'eigenvalue':values, 'residual':[eigenpair_residual(A,x,vectors[:,i]) for i,x in enumerate(values)]}))
print('Negative and zero eigenvalue examples:',np.diag([-2.,0.])@np.eye(2))

In [ ]:
original=np.array([[1.,0.],[0.,1.],[1.,1.]])
transformed=original@A.T
fig,axes=plt.subplots(1,3,figsize=(12,4),layout='constrained')
for ax,v,w,label in zip(axes,original,transformed,['First axis','Second axis','Not an eigenvector']):
    ax.quiver(0,0,*v,angles='xy',scale_units='xy',scale=1,color='#1764a0',label='Original',width=.015)
    ax.quiver(0,0,*w,angles='xy',scale_units='xy',scale=1,color='#d97706',alpha=.65,label='Transformed',width=.025)
    ax.set(xlim=(-.3,3.5),ylim=(-.3,3.5),title=label,xlabel='Component 1',ylabel='Component 2')
    ax.set_aspect('equal'); ax.grid(alpha=.2); ax.legend(fontsize=8)
plt.show()

## 2. Symmetry gives an orthonormal eigenbasis
Take
$$
S=\begin{pmatrix}2&1\\1&2\end{pmatrix}.
$$
The characteristic polynomial is $(2-\lambda)^2-1=\lambda^2-4\lambda+3$, so the eigenvalues are $3$ and $1$. Corresponding unit eigenvectors are $(1,1)^T/\sqrt2$ and $(1,-1)^T/\sqrt2$. Numerical libraries can reverse either sign.

For a **real symmetric** matrix, the spectral theorem gives $S=QDQ^T$ with $Q^TQ=I$. With repeated eigenvalues, the eigenbasis within a repeated eigenspace is not unique. This theorem does not apply to arbitrary nonsymmetric real matrices.

In [ ]:
S=np.array([[2.,1.],[1.,2.]])
s_values,Q=symmetric_eigendecomposition(S)
assert np.allclose(s_values,[3.,1.])
assert np.allclose(Q.T@Q,np.eye(2))
assert np.allclose(Q@np.diag(s_values)@Q.T,S)
assert all(eigenpair_residual(S,x,Q[:,i])<1e-12 for i,x in enumerate(s_values))
print('Eigenvalues:',s_values,'\nOrthonormal eigenvectors (columns):\n',Q)

## 3. Diagonalization, powers, and a counterexample
If a square matrix has a complete eigenvector basis over the chosen field, then
$$
A=PDP^{-1},\qquad A^k=PD^kP^{-1}.
$$
For our symmetric example,
$$
S^k=\frac12\begin{pmatrix}3^k+1&3^k-1\\3^k-1&3^k+1\end{pmatrix}.
$$
This holds also for $k=0$. Determinants are useful for small hand derivations; numerical eigensolvers are preferable to forming a large characteristic polynomial.

Not every matrix is diagonalizable. The Jordan matrix $J$ below has eigenvalue $1$ twice but only one independent eigenvector. Its powers contain a growing off-diagonal entry. Near-defective matrices can also yield ill-conditioned eigenvector matrices even when diagonalization is theoretically possible.

In [ ]:
P,D,P_inv=diagonalize(S)
assert np.allclose(P@D@P_inv,S)
for k in [0,1,2,10]:
    expected=.5*np.array([[3**k+1,3**k-1],[3**k-1,3**k+1]])
    assert np.allclose(matrix_power_via_eigendecomposition(S,k),expected)
    assert np.allclose(np.linalg.matrix_power(S,k),expected)
print('S^10 =\n',np.linalg.matrix_power(S,10))
J=np.array([[1.,1.],[0.,1.]])
assert np.linalg.matrix_rank(J-np.eye(2))==1
assert np.allclose(np.linalg.matrix_power(J,10),[[1,10],[0,1]])
try:
    diagonalize(J)
except ValueError:
    print('Defective Jordan matrix correctly rejected by diagonalize')
else:
    raise AssertionError('Unexpected diagonalization of defective matrix')
R=np.array([[0.,-1.],[1.,0.]])
rv,rq=eigendecomposition(R)
assert np.allclose(np.sort_complex(rv),np.sort_complex([1j,-1j]))
assert all(eigenpair_residual(R,x,rq[:,i])<1e-12 for i,x in enumerate(rv))
print('Real rotation, complex eigenvalues:',rv)

## 4. Power iteration: verify the residual, not just a stable scalar
Repeated multiplication and normalization often select a dominant eigendirection. Standard convergence needs a unique eigenvalue of largest modulus and a suitable initial component, with additional care for nonsymmetric/defective problems. A negative dominant eigenvalue can make vector signs alternate; a residual-based test handles that.

The bundled runtime's legacy `power_iteration` stops on changes in the Rayleigh quotient and its `verify_eigenpair` does not reject zero vectors. **This lesson therefore uses the explicit reference functions above for acceptance**, without changing or relabelling the package. These are small dense teaching routines, not production sparse solvers.

For $\operatorname{diag}(1,-1)$ and initial vector $(1,1)$, the Rayleigh quotient remains zero while the vector is not an eigenvector. Conversely, starting on a nondominant eigenvector gives residual zero immediately: residual convergence alone cannot establish dominance.

In [ ]:
dominant=residual_power_iteration(S,initial=[1.,.2])
assert np.isclose(dominant['value'],3.) and dominant['residual']<=1e-10
print('S dominant estimate:',dominant)
negative=residual_power_iteration(np.diag([-3.,1.]),initial=[1.,1.])
assert np.isclose(negative['value'],-3.)
assert negative['residual']<=1e-10
print('Negative dominant eigenvalue:',negative['value'])
try:
    residual_power_iteration(np.diag([1.,-1.]),initial=[1.,1.],max_iter=30)
except RuntimeError:
    print('Equal-modulus oscillation: correctly refused false convergence')
else:
    raise AssertionError('False convergence accepted')
hidden=residual_power_iteration(A,initial=[0.,1.])
assert np.isclose(hidden['value'],1.)
print('An exact but NONdominant eigenpair from a special start:',hidden['value'])

## 5. Spectral radius and discrete-time stability
The spectral radius is
$$
\rho(A)=\max_i|\lambda_i|.
$$
For a finite-dimensional **autonomous discrete-time linear system** $x_{t+1}=Ax_t$, every initial state tends to zero exactly when $\rho(A)<1$. This is not the same criterion as for continuous-time dynamics. Inputs, time-varying coefficients and nonlinearities require other analysis.

At radius one, the identity keeps states constant while a nontrivial Jordan block can cause growth. Even below one, a nonnormal matrix may show substantial transient amplification before eventual decay. A single decaying trajectory cannot establish stability for all initial states.

In [ ]:
stable=np.array([[.7,.1],[0.,.8]])
unstable=np.diag([1.1,.9])
transient=np.array([[.8,4.],[0.,.8]])
def trajectory(matrix,initial,steps=60):
    path=[np.asarray(initial,dtype=float)]
    for _ in range(steps): path.append(matrix@path[-1])
    return np.asarray(path)
assert np.isclose(spectral_radius(stable),.8)
assert np.isclose(spectral_radius(unstable),1.1)
assert np.isclose(spectral_radius(J),1.)
assert np.isclose(spectral_radius(transient),.8)
p_stable=trajectory(stable,[1,1]); p_unstable=trajectory(unstable,[1,1])
p_transient=trajectory(transient,[0,1])
assert np.allclose(p_stable,.8**np.arange(61)[:,None]*np.ones((1,2)))
assert np.linalg.norm(p_transient,axis=1).max()>5
assert np.linalg.norm(p_transient[-1])<.001
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for path,label in [(p_stable,'Stable: radius 0.8'),(p_unstable,'Unstable: radius 1.1')]:
    axes[0].plot(np.linalg.norm(path,axis=1),label=label)
axes[0].set_yscale('log'); axes[0].set_title('Asymptotic contrast (log norm)')
for matrix,label in [(np.eye(2),'Identity: radius 1'),(J,'Jordan: radius 1'),(transient,'Transient growth: radius 0.8')]:
    axes[1].plot(np.linalg.norm(trajectory(matrix,[0,1]),axis=1),label=label)
axes[1].set_title('Boundary behavior and transient growth')
for ax in axes:
    ax.set_xlabel('Time step');ax.set_ylabel('State-vector norm');ax.legend(fontsize=8);ax.grid(alpha=.2)
plt.show()

## 6. Statistics: covariance eigenvectors and principal components
Center each column before forming sample covariance, using divisor $n-1$. For a unit vector $q$, the sample variance of centered scores $X_cq$ equals $q^TCq$. Covariance eigenvectors identify orthogonal directions of variation, and eigenvalues quantify their variances.

The following four observations illustrate arithmetic only. Scaling a measured variable changes covariance PCA; correlation/standardized PCA answers a different question. High explained variance does not prove prediction accuracy, representativeness, causality or decision usefulness.

In [ ]:
X=np.array([[2.,1.],[3.,2.],[4.,2.5],[5.,4.]])
Xc=X-X.mean(axis=0)
C=Xc.T@Xc/(len(X)-1)
c_values,c_vectors=symmetric_eigendecomposition(C)
ratios=c_values/c_values.sum()
assert np.allclose(C,[[5/3,19/12],[19/12,25/16]])
assert np.allclose(c_values,[3.19877307,.03039360],atol=1e-8)
assert np.isclose(ratios.sum(),1.) and (c_values>=0).all()
scores=Xc@c_vectors
assert np.allclose(np.var(scores,axis=0,ddof=1),c_values)
assert np.allclose(scores@c_vectors.T,Xc)
display(pd.DataFrame({'eigenvalue':c_values,'explained_variance_ratio':ratios}))
X_scaled=X*np.array([1.,100.])
scaled_values,_=symmetric_eigendecomposition(np.cov(X_scaled,rowvar=False))
print('After multiplying variable 2 by 100:',scaled_values/scaled_values.sum())

## 7. Decision interpretation: a synthetic sector propagation model
We posit $x_{t+1}=Mx_t$, where entry $M_{ij}$ maps component $j$ into component $i$. These coefficients are invented for teaching; they are not fitted to observations. Rows and columns do **not** sum to one: this is not a Markov transition-probability matrix.

This strictly positive matrix has a positive dominant eigendirection. Normalizing that eigenvector to sum one makes its components easier to compare, but does not turn them into budget allocations or causal importance. Its dominant eigenvalue exceeds one, implying amplification along that mode within this hypothetical autonomous model.

In [ ]:
sectors=['Agriculture','Health','Energy','Transport']
M=np.array([[.70,.10,.20,.15],[.10,.80,.25,.10],[.25,.10,.75,.30],[.20,.15,.35,.70]])
result=residual_power_iteration(M)
v=result['vector']
if v.sum()<0: v=-v
assert (v>0).all()
weights=v/v.sum()
assert np.isclose(result['value'],1.3114820544971353)
assert np.isclose(result['value'],spectral_radius(M))
assert eigenpair_residual(M,result['value'],weights)<1e-10
assert np.allclose(weights,[.20024812,.23284988,.28255681,.28434520],atol=1e-8)
assert not np.allclose(M.sum(axis=0),1.) and not np.allclose(M.sum(axis=1),1.)
display(pd.DataFrame({'synthetic_dominant_mode_weight':weights},index=sectors))
print('Spectral radius:',spectral_radius(M),'Residual:',result['residual'])
print('NOT empirical priorities, probabilities or budget shares.')

## 8. Exercises — attempt these before the solutions
1. For $\operatorname{diag}(-2,0)$, identify two eigenpairs and interpret them. Why is the zero vector excluded?
2. Derive the characteristic polynomial of $S$, and find an orthonormal eigenbasis.
3. Compute $S^{10}$ by its eigenbasis. What happens for power zero?
4. Prove $J^k=\begin{pmatrix}1&k\\0&1\end{pmatrix}$. Why can $J$ not be diagonalized?
5. Explain why equal successive Rayleigh quotients do not imply convergence for $\operatorname{diag}(1,-1)$.
6. Give a starting vector that hides the unstable mode of $\operatorname{diag}(1.1,0.9)$.
7. Explain why radius $0.8$ does not rule out short-term amplification.
8. Compute the two sample covariance eigenvalues and explain the denominator $n-1$.
9. Interpret the four-sector weights and list three reasons they cannot determine expenditure priorities.
10. State the difference between an approximate eigenpair residual check and proof of dominance.

## 9. Worked solutions and interpretation
1. $(\lambda,v)=(-2,(1,0)^T)$ reverses orientation and doubles length; $(0,(0,1)^T)$ maps to zero. Allowing $v=0$ would satisfy the equation for every scalar and identify no direction.
2. $\det(S-\lambda I)=(2-\lambda)^2-1=(\lambda-3)(\lambda-1)$. Normalize $(1,1)^T$ and $(1,-1)^T$ by $\sqrt2$; they are orthogonal.
3. The diagonal entries are $(59049+1)/2=29525$ and off-diagonal entries $(59049-1)/2=29524$. Power zero yields the identity.
4. Write $J=I+N$ with $N^2=0$; the binomial expansion gives $I+kN$. The eigenspace for eigenvalue one has dimension one, not two.
5. Equal-magnitude components give Rayleigh quotient zero while the sign of one component alternates. The eigenpair residual remains nonzero.
6. Starting at $(0,1)^T$ produces decay as $0.9^t$, despite the unstable eigenvalue $1.1$. Stability for all starts is a stronger claim.
7. For the upper-triangular example, coupling feeds the second component into the first. The off-diagonal term in its $k$th power is $4k(0.8)^{k-1}$, which can initially increase but eventually vanishes.
8. The covariance eigenvalues are approximately $3.19877307$ and $0.03039360$. Sample covariance uses $n-1$ after estimating the mean; this is not a claim that four observations support reliable statistical inference.
9. The normalized components describe one model eigendirection. Coefficients are synthetic, causal identification is absent, and costs/objectives/constraints are not modeled. All three prevent interpreting weights as allocations.
10. Any eigenpair can have residual zero, including a nondominant one. Dominance requires spectral comparison or justified convergence assumptions; the sector example also compares against the dense eigensolver.

## 10. Final acceptance and next steps
The next cell checks the main lesson outputs. Keep the recorded environment with any reported result. A PASS does not certify all functions of the bundled library. The chapter, video, independent review, live Colab/VS Code acceptance and public release remain separate work items.

In [ ]:
checks={
    'symmetric_reconstruction': bool(np.allclose(Q@np.diag(s_values)@Q.T,S)),
    'power_10': bool(np.allclose(np.linalg.matrix_power(S,10),[[29525,29524],[29524,29525]])),
    'covariance_score_variance': bool(np.allclose(np.var(scores,axis=0,ddof=1),c_values)),
    'sector_residual': bool(result['residual']<=1e-10),
    'sector_growth_disclosed': bool(result['value']>1),
}
assert all(checks.values())
print('LESSON 6 NOTEBOOK CHECKS: PASS')
print(checks)
print('Runtime:',srai_math.__version__,'; Python:',sys.version.split()[0])
print('Candidate only; external publication and live UI checks are not asserted.')